Loading the raw data set

In [2]:
import pandas as pd

df = pd.read_excel("../data/raw/Online Retail.xlsx")
print(df.head())


  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  


Quick inspection

In [3]:
print(df.shape)
print(df.isnull().sum())

(541909, 8)
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


Checking missing values

In [4]:
df[df["CustomerID"].isnull()].head()
df[df["Description"].isnull()].head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom


Dropping rows with missing customer IDs and descriptions

In [5]:
print(df["CustomerID"].isnull().sum())
print(df["Description"].isnull().sum())


135080
1454


In [6]:

df = df.dropna(subset=["CustomerID"])
df = df.dropna(subset=["Description"])
print(df["CustomerID"].isnull().sum())
print(df["Description"].isnull().sum())

0
0


Identifying canceled orders

In [7]:
df["InvoiceNo"] = df["InvoiceNo"].astype(str)

cancelled = df[df["InvoiceNo"].str.startswith("C")]
print(cancelled.head())
print(cancelled.shape)

    InvoiceNo StockCode                       Description  Quantity  \
141   C536379         D                          Discount        -1   
154   C536383    35004C   SET OF 3 COLOURED  FLYING DUCKS        -1   
235   C536391     22556    PLASTERS IN TIN CIRCUS PARADE        -12   
236   C536391     21984  PACK OF 12 PINK PAISLEY TISSUES        -24   
237   C536391     21983  PACK OF 12 BLUE PAISLEY TISSUES        -24   

            InvoiceDate  UnitPrice  CustomerID         Country  
141 2010-12-01 09:41:00      27.50     14527.0  United Kingdom  
154 2010-12-01 09:49:00       4.65     15311.0  United Kingdom  
235 2010-12-01 10:24:00       1.65     17548.0  United Kingdom  
236 2010-12-01 10:24:00       0.29     17548.0  United Kingdom  
237 2010-12-01 10:24:00       0.29     17548.0  United Kingdom  
(8905, 8)


Dropping cancelled orders

In [8]:
df = df[~df["InvoiceNo"].str.startswith("C")]
print(df["InvoiceNo"].str.startswith("C").sum())


0


Checking and dropping quantities which are 0 or less than 0

In [9]:
print((df["Quantity"] <= 0).sum())
print(df[df["Quantity"] <= 0].head())

0
Empty DataFrame
Columns: [InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country]
Index: []


Checking and dropping prices which are 0 or less than 0

In [10]:
print((df["UnitPrice"] <= 0).sum())
print(df[df["UnitPrice"] <= 0].head())

40
      InvoiceNo StockCode                   Description  Quantity  \
9302     537197     22841  ROUND CAKE TIN VINTAGE GREEN         1   
33576    539263     22580  ADVENT CALENDAR GINGHAM SACK         4   
40089    539722     22423      REGENCY CAKESTAND 3 TIER        10   
47068    540372     22090       PAPER BUNTING RETROSPOT        24   
47070    540372     22553        PLASTERS IN TIN SKULLS        24   

              InvoiceDate  UnitPrice  CustomerID         Country  
9302  2010-12-05 14:02:00        0.0     12647.0         Germany  
33576 2010-12-16 14:36:00        0.0     16560.0  United Kingdom  
40089 2010-12-21 13:45:00        0.0     14911.0            EIRE  
47068 2011-01-06 16:41:00        0.0     13081.0  United Kingdom  
47070 2011-01-06 16:41:00        0.0     13081.0  United Kingdom  


In [11]:
df = df[df["UnitPrice"] > 0]
print((df["UnitPrice"] <= 0).sum())
print(df[df["UnitPrice"] <= 0].head())

0
Empty DataFrame
Columns: [InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country]
Index: []


Converting data types

In [12]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["CustomerID"] = df["CustomerID"].astype(str)

Adding new usefull columns

In [13]:
df["Revenue"] = df["Quantity"] * df["UnitPrice"]
df["InvoiceMonth"] = df["InvoiceDate"].dt.to_period("M").astype(str)
df["InvoiceDay"] = df["InvoiceDate"].dt.date
df["InvoiceHour"] = df["InvoiceDate"].dt.hour
df["Weekday"] = df["InvoiceDate"].dt.day_name()

Checking the new clean data

In [14]:
print(df.shape)
print(df.info())
print(df.head())
print(df.describe())
print(df.isnull().sum())

(397884, 13)
<class 'pandas.core.frame.DataFrame'>
Index: 397884 entries, 0 to 541908
Data columns (total 13 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   InvoiceNo     397884 non-null  object        
 1   StockCode     397884 non-null  object        
 2   Description   397884 non-null  object        
 3   Quantity      397884 non-null  int64         
 4   InvoiceDate   397884 non-null  datetime64[ns]
 5   UnitPrice     397884 non-null  float64       
 6   CustomerID    397884 non-null  object        
 7   Country       397884 non-null  object        
 8   Revenue       397884 non-null  float64       
 9   InvoiceMonth  397884 non-null  object        
 10  InvoiceDay    397884 non-null  object        
 11  InvoiceHour   397884 non-null  int32         
 12  Weekday       397884 non-null  object        
dtypes: datetime64[ns](1), float64(2), int32(1), int64(1), object(8)
memory usage: 41.0+ MB
None
  InvoiceNo StockCo

Saving the new clean file

In [15]:
df.to_csv("../data/cleaned/online_retail_cleaned.csv", index=False)